# Quantum Reasoning Co-Processor — GPU Qubit Scaling (Extended)
**Push to T4 VRAM limit: 18q → 21q → 24q → 27q**

Author: Saikrishna Garikipati

> **Runtime → Run all** — runs end-to-end unattended. **GPU REQUIRED** (T4/A100).

Run 10 showed +4% advantage at 18 qubits (single seed). This notebook:
1. **Confirms 18q** with 5 seeds (is the signal real?)
2. **Extends to 21q** (register_dim=128, 3 seeds)
3. **Pushes to 24q** (register_dim=256, 2 seeds)
4. **Stretch: 27q** (register_dim=512, 1 seed — near T4 VRAM limit)

If advantage grows: 18q → 21q → 24q → 27q, the scaling thesis is definitively validated.

In [ ]:
##############################################################################
# STEP 1: SETUP & GPU VERIFICATION
##############################################################################
import os, sys, time
print('=' * 60)
print('STEP 1: SETUP & GPU VERIFICATION')
print('=' * 60)

!pip install -q pennylane pennylane-lightning 'pennylane-lightning[gpu]' autograd scipy 2>&1 | tail -3

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import autograd
import autograd.numpy as anp
import pennylane as qml
import pennylane.numpy as pnp

print(f'PennyLane {qml.__version__}')

# --- HARD GPU CHECK ---
try:
    test_dev = qml.device('lightning.gpu', wires=4)
    print(f'✓ lightning.gpu AVAILABLE')
    del test_dev
except Exception as e:
    print(f'✗ lightning.gpu NOT available: {e}')
    print()
    print('FATAL: This notebook REQUIRES GPU runtime.')
    print('Fix: Runtime → Change runtime type → GPU (T4 or A100)')
    print('Then: Restart runtime and run all again.')
    raise SystemExit('GPU required. Process killed.')

print('\nSetup complete. GPU ready.')

In [ ]:
##############################################################################
# STEP 2: DEFINE QUANTUM ATTENTION CIRCUIT
##############################################################################
print('=' * 60)
print('STEP 2: QUANTUM ATTENTION CIRCUIT')
print('=' * 60)

def build_attention_circuit(n_qubits, n_layers=2):
    """Build Q-K→V quantum attention circuit on GPU.
    Returns SINGLE SCALAR (mean PauliZ on V-register) for autograd compatibility.
    lightning.gpu + adjoint cannot return multiple expvals through autograd.grad().
    """
    assert n_qubits % 3 == 0
    qpr = n_qubits // 3
    q_wires = list(range(0, qpr))
    k_wires = list(range(qpr, 2 * qpr))
    v_wires = list(range(2 * qpr, 3 * qpr))

    dev = qml.device('lightning.gpu', wires=n_qubits)

    # Hamiltonian: average of PauliZ on V-register → single scalar output
    coeffs = [1.0 / qpr] * qpr
    obs = [qml.PauliZ(w) for w in v_wires]
    H = qml.Hamiltonian(coeffs, obs)

    @qml.qnode(dev, interface='autograd', diff_method='adjoint')
    def circuit(q_features, k_features, v_features, params):
        qml.AmplitudeEmbedding(q_features, wires=q_wires, normalize=True)
        qml.AmplitudeEmbedding(k_features, wires=k_wires, normalize=True)
        qml.AmplitudeEmbedding(v_features, wires=v_wires, normalize=True)
        idx = 0
        for layer in range(n_layers):
            for i in range(qpr):
                qml.CNOT(wires=[q_wires[i], k_wires[i]])
            for i in range(qpr):
                qml.CNOT(wires=[k_wires[i], v_wires[i]])
                qml.RY(params[idx], wires=v_wires[i])
                qml.CNOT(wires=[k_wires[i], v_wires[i]])
                idx += 1
            for i in range(qpr):
                qml.RY(params[idx], wires=v_wires[i])
                qml.RZ(params[idx + 1], wires=v_wires[i])
                idx += 2
            for i in range(qpr - 1):
                qml.CNOT(wires=[v_wires[i], v_wires[i + 1]])
            for i in range(qpr):
                qml.CNOT(wires=[q_wires[i], k_wires[i]])
        return qml.expval(H)

    n_params = n_layers * (qpr + qpr * 2)
    params = pnp.array(np.random.randn(n_params) * 0.5, requires_grad=True)
    print(f'  {n_qubits}q circuit: {n_params} params, register_dim={2**qpr}')
    print(f'  Returns: single scalar (mean PauliZ on V-register)')
    return circuit, params, qpr

# Quick verification
test_circuit, test_params, _ = build_attention_circuit(12)
q_test = np.random.randn(16)
out = test_circuit(q_test, q_test, q_test, test_params)
print(f'  12q forward pass OK: output={float(out):.4f} (scalar in [-1,1])')
print('Circuit definition ready.')

In [ ]:
##############################################################################
# STEP 3: DEFINE CLASSICAL BASELINE
##############################################################################
print('=' * 60)
print('STEP 3: CLASSICAL BASELINE (matched params)')
print('=' * 60)

def build_classical_model(input_dim, output_dim, n_quantum_params):
    """Classical MLP matched to quantum param count."""
    hidden = max(output_dim, n_quantum_params // 3)
    W1 = pnp.array(np.random.randn(input_dim * 3, hidden) * 0.1, requires_grad=True)
    b1 = pnp.array(np.zeros(hidden), requires_grad=True)
    W2 = pnp.array(np.random.randn(hidden, output_dim) * 0.1, requires_grad=True)
    b2 = pnp.array(np.zeros(output_dim), requires_grad=True)
    total = W1.size + b1.size + W2.size + b2.size
    print(f'  Classical: {total} params (hidden={hidden})')
    return [W1, b1, W2, b2]

def classical_forward(q, k, v, params):
    W1, b1, W2, b2 = params
    x = anp.concatenate([q, k, v])
    h = anp.maximum(x @ W1 + b1, 0)
    return anp.tanh(W2.T @ h + b2)

print('Classical model defined.')

In [ ]:
##############################################################################
# STEP 4: DATA GENERATION
##############################################################################
print('=' * 60)
print('STEP 4: XOR-SIGN CLASSIFICATION TASK')
print('=' * 60)

def generate_xor_sign_data(dim, n_samples, seed=42):
    """
    Task: Predict XOR of sign-agreements between Q and K on hidden dims.
    This directly exploits CNOT(Q,K) → |K⊕Q⟩ mechanism.
    """
    rng = np.random.RandomState(seed)
    n_hidden = min(dim // 2, 4)
    hidden_dims = sorted(rng.choice(dim, n_hidden, replace=False).tolist())
    Q = rng.randn(n_samples, dim).astype(np.float64)
    K = rng.randn(n_samples, dim).astype(np.float64)
    V = rng.randn(n_samples, dim).astype(np.float64)
    agreements = np.array([(np.sign(Q[:, i]) == np.sign(K[:, i])).astype(int)
                           for i in hidden_dims])
    labels = np.bitwise_xor.reduce(agreements, axis=0)
    return Q, K, V, labels, hidden_dims

# Verify
Q, K, V, y, hd = generate_xor_sign_data(16, 100, seed=42)
print(f'  Example: dim=16, samples=100, hidden={hd}, balance={y.mean():.2f}')
print('Data generation ready.')

In [ ]:
##############################################################################
# STEP 5: TRAINING FUNCTIONS
##############################################################################
print('=' * 60)
print('STEP 5: TRAINING INFRASTRUCTURE')
print('=' * 60)

def train_quantum_circuit(circuit, params, Q, K, V, y, epochs=30, lr=0.01, batch_size=32):
    n = len(Q)
    for epoch in range(epochs):
        idx = np.random.permutation(n)
        epoch_loss = 0.0
        nb = 0
        for start in range(0, min(n, 200) - batch_size + 1, batch_size):
            batch = idx[start:start + batch_size]
            def loss_fn(p):
                total = 0.0
                for j in batch:
                    score = circuit(Q[j], K[j], V[j], p)
                    pred = (score + 1) / 2
                    t = float(y[j])
                    total = total - (t * anp.log(pred + 1e-10) + (1-t) * anp.log(1-pred + 1e-10))
                return total / len(batch)
            val_and_grad = autograd.value_and_grad(loss_fn)
            loss_val, grads = val_and_grad(params)
            epoch_loss += float(loss_val)
            params = pnp.array(params - lr * np.array(grads), requires_grad=True)
            nb += 1
        if (epoch + 1) % 10 == 0 or epoch == 0:
            acc = eval_quantum(circuit, params, Q[:200], K[:200], V[:200], y[:200])
            print(f'    Epoch {epoch+1:2d}: loss={epoch_loss/max(nb,1):.4f}, acc={acc:.1%}')
    return params

def train_classical_model(params, Q, K, V, y, epochs=30, lr=0.01, batch_size=32):
    n = len(Q)
    for epoch in range(epochs):
        idx = np.random.permutation(n)
        epoch_loss = 0.0
        nb = 0
        for start in range(0, min(n, 200) - batch_size + 1, batch_size):
            batch = idx[start:start + batch_size]
            def loss_fn(flat):
                sizes = [p.size for p in params]
                shapes = [p.shape for p in params]
                ps = []
                offset = 0
                for s, sh in zip(sizes, shapes):
                    ps.append(flat[offset:offset+s].reshape(sh))
                    offset += s
                total = 0.0
                for j in batch:
                    out = classical_forward(Q[j], K[j], V[j], ps)
                    score = anp.mean(out)
                    pred = (score + 1) / 2
                    t = float(y[j])
                    total = total - (t * anp.log(pred + 1e-10) + (1-t) * anp.log(1-pred + 1e-10))
                return total / len(batch)
            flat = np.concatenate([np.array(p).flatten() for p in params])
            val_and_grad = autograd.value_and_grad(loss_fn)
            loss_val, grads = val_and_grad(flat)
            epoch_loss += float(loss_val)
            flat = flat - lr * np.array(grads)
            offset = 0
            for k_idx, p in enumerate(params):
                s = p.size
                params[k_idx] = pnp.array(flat[offset:offset+s].reshape(p.shape), requires_grad=True)
                offset += s
            nb += 1
        if (epoch + 1) % 10 == 0 or epoch == 0:
            acc = eval_classical(params, Q[:200], K[:200], V[:200], y[:200])
            print(f'    Epoch {epoch+1:2d}: loss={epoch_loss/max(nb,1):.4f}, acc={acc:.1%}')
    return params

def eval_quantum(circuit, params, Q, K, V, y):
    correct = sum(1 for j in range(len(Q))
                  if (1 if float(circuit(Q[j], K[j], V[j], params)) > 0 else 0) == int(y[j]))
    return correct / len(Q)

def eval_classical(params, Q, K, V, y):
    correct = sum(1 for j in range(len(Q))
                  if (1 if float(np.mean(classical_forward(Q[j], K[j], V[j], params))) > 0 else 0) == int(y[j]))
    return correct / len(Q)

print('Training functions ready.')

In [ ]:
##############################################################################
# STEP 6: MULTI-SEED 18-QUBIT CONFIRMATION
##############################################################################
print('=' * 60)
print('STEP 6: 18-QUBIT CONFIRMATION (5 seeds)')
print('=' * 60)
print('Run 10 showed +4% at 18q with seed=42. Confirming with 5 seeds...')

RESULTS = {}
SEEDS = [42, 99, 137, 256, 314]

dim18 = 2 ** (18 // 3)  # 64
n_hidden_18 = min(dim18 // 8, 6)  # scale hidden dims with input size

q18_accs, c18_accs = [], []
for seed in SEEDS:
    np.random.seed(seed)
    Q, K, V, y, hd = generate_xor_sign_data(dim18, 500, seed=seed)
    print(f'\n  Seed {seed} (hidden={hd}, balance={y.mean():.2f}):')

    # Quantum
    t0 = time.perf_counter()
    circ, params, _ = build_attention_circuit(18, n_layers=2)
    params = train_quantum_circuit(circ, params, Q[:400], K[:400], V[:400], y[:400], epochs=30)
    q_acc = eval_quantum(circ, params, Q[400:], K[400:], V[400:], y[400:])
    q_time = time.perf_counter() - t0
    q18_accs.append(q_acc)

    # Classical
    t0 = time.perf_counter()
    cp = build_classical_model(dim18, 18//3, params.size)
    cp = train_classical_model(cp, Q[:400], K[:400], V[:400], y[:400], epochs=30)
    c_acc = eval_classical(cp, Q[400:], K[400:], V[400:], y[400:])
    c18_accs.append(c_acc)

    print(f'    Quantum: {q_acc:.1%} | Classical: {c_acc:.1%} | Adv: {(q_acc-c_acc)*100:+.1f}% ({q_time:.0f}s)')

mean_q = np.mean(q18_accs)
mean_c = np.mean(c18_accs)
adv_18 = mean_q - mean_c
print(f'\n  18q MEAN: Quantum={mean_q:.1%} ± {np.std(q18_accs):.1%} | Classical={mean_c:.1%} | Advantage={adv_18*100:+.1f}%')
RESULTS[18] = {'q_accs': q18_accs, 'c_accs': c18_accs, 'mean_q': mean_q, 'mean_c': mean_c, 'advantage': adv_18}

In [ ]:
##############################################################################
# STEP 7: 21-QUBIT EXPERIMENT (3 seeds)
##############################################################################
print('=' * 60)
print('STEP 7: 21-QUBIT (register_dim=128, 3 seeds)')
print('=' * 60)

dim21 = 2 ** (21 // 3)  # 128

q21_accs, c21_accs = [], []
for seed in SEEDS[:3]:
    np.random.seed(seed)
    Q, K, V, y, hd = generate_xor_sign_data(dim21, 500, seed=seed)
    print(f'\n  Seed {seed} (hidden={hd}, balance={y.mean():.2f}):')

    t0 = time.perf_counter()
    circ, params, _ = build_attention_circuit(21, n_layers=2)
    params = train_quantum_circuit(circ, params, Q[:400], K[:400], V[:400], y[:400], epochs=30)
    q_acc = eval_quantum(circ, params, Q[400:], K[400:], V[400:], y[400:])
    q_time = time.perf_counter() - t0
    q21_accs.append(q_acc)

    t0 = time.perf_counter()
    cp = build_classical_model(dim21, 21//3, params.size)
    cp = train_classical_model(cp, Q[:400], K[:400], V[:400], y[:400], epochs=30)
    c_acc = eval_classical(cp, Q[400:], K[400:], V[400:], y[400:])
    c21_accs.append(c_acc)

    print(f'    Quantum: {q_acc:.1%} | Classical: {c_acc:.1%} | Adv: {(q_acc-c_acc)*100:+.1f}% ({q_time:.0f}s)')

mean_q = np.mean(q21_accs)
mean_c = np.mean(c21_accs)
adv_21 = mean_q - mean_c
print(f'\n  21q MEAN: Quantum={mean_q:.1%} ± {np.std(q21_accs):.1%} | Classical={mean_c:.1%} | Advantage={adv_21*100:+.1f}%')
RESULTS[21] = {'q_accs': q21_accs, 'c_accs': c21_accs, 'mean_q': mean_q, 'mean_c': mean_c, 'advantage': adv_21}

In [ ]:
##############################################################################
# STEP 8: 24-QUBIT EXPERIMENT (2 seeds)
##############################################################################
print('=' * 60)
print('STEP 8: 24-QUBIT (register_dim=256, 2 seeds)')
print('=' * 60)
print('State vector: 2^24 = 16.7M amplitudes (~256MB). GPU handles this.')

dim24 = 2 ** (24 // 3)  # 256

q24_accs, c24_accs = [], []
for seed in SEEDS[:2]:
    np.random.seed(seed)
    Q, K, V, y, hd = generate_xor_sign_data(dim24, 500, seed=seed)
    print(f'\n  Seed {seed} (hidden={hd}, balance={y.mean():.2f}):')

    t0 = time.perf_counter()
    circ, params, _ = build_attention_circuit(24, n_layers=2)
    params = train_quantum_circuit(circ, params, Q[:400], K[:400], V[:400], y[:400], epochs=20, lr=0.02)
    q_acc = eval_quantum(circ, params, Q[400:], K[400:], V[400:], y[400:])
    q_time = time.perf_counter() - t0
    q24_accs.append(q_acc)

    t0 = time.perf_counter()
    cp = build_classical_model(dim24, 24//3, params.size)
    cp = train_classical_model(cp, Q[:400], K[:400], V[:400], y[:400], epochs=20, lr=0.02)
    c_acc = eval_classical(cp, Q[400:], K[400:], V[400:], y[400:])
    c24_accs.append(c_acc)

    print(f'    Quantum: {q_acc:.1%} | Classical: {c_acc:.1%} | Adv: {(q_acc-c_acc)*100:+.1f}% ({q_time:.0f}s)')

mean_q = np.mean(q24_accs)
mean_c = np.mean(c24_accs)
adv_24 = mean_q - mean_c
print(f'\n  24q MEAN: Quantum={mean_q:.1%} ± {np.std(q24_accs):.1%} | Classical={mean_c:.1%} | Advantage={adv_24*100:+.1f}%')
RESULTS[24] = {'q_accs': q24_accs, 'c_accs': c24_accs, 'mean_q': mean_q, 'mean_c': mean_c, 'advantage': adv_24}

In [ ]:
##############################################################################
# STEP 9: 27-QUBIT STRETCH GOAL (1 seed)
##############################################################################
print('=' * 60)
print('STEP 9: 27-QUBIT STRETCH (register_dim=512, 1 seed)')
print('=' * 60)
print('State vector: 2^27 = 134M amplitudes (~2GB). Near T4 limit.')
print('Adjoint needs ~3x = ~6-7GB. Should fit in 16GB VRAM.')

dim27 = 2 ** (27 // 3)  # 512
seed = 42

try:
    np.random.seed(seed)
    Q, K, V, y, hd = generate_xor_sign_data(dim27, 300, seed=seed)  # fewer samples for speed
    print(f'  Data: dim={dim27}, n=300, hidden={hd}, balance={y.mean():.2f}')

    print('\n  Quantum (27q):')
    t0 = time.perf_counter()
    circ, params, _ = build_attention_circuit(27, n_layers=2)
    params = train_quantum_circuit(circ, params, Q[:200], K[:200], V[:200], y[:200],
                                   epochs=15, lr=0.03, batch_size=16)
    q_acc = eval_quantum(circ, params, Q[200:], K[200:], V[200:], y[200:])
    q_time = time.perf_counter() - t0
    print(f'  Quantum acc: {q_acc:.1%} ({q_time:.0f}s)')

    print('\n  Classical (matched):')
    t0 = time.perf_counter()
    cp = build_classical_model(dim27, 27//3, params.size)
    cp = train_classical_model(cp, Q[:200], K[:200], V[:200], y[:200],
                               epochs=15, lr=0.03, batch_size=16)
    c_acc = eval_classical(cp, Q[200:], K[200:], V[200:], y[200:])
    c_time = time.perf_counter() - t0
    print(f'  Classical acc: {c_acc:.1%} ({c_time:.0f}s)')

    adv_27 = q_acc - c_acc
    print(f'\n  27q ADVANTAGE: {adv_27*100:+.1f}%')
    RESULTS[27] = {'q_accs': [q_acc], 'c_accs': [c_acc], 'mean_q': q_acc, 'mean_c': c_acc, 'advantage': adv_27}

except Exception as e:
    print(f'\n  27q FAILED (likely OOM): {e}')
    print('  This is expected if T4 VRAM is insufficient for 27-qubit adjoint.')
    RESULTS[27] = {'q_accs': [], 'c_accs': [], 'mean_q': 0, 'mean_c': 0, 'advantage': 0, 'failed': True}

In [ ]:
##############################################################################
# STEP 10: FINAL SCALING REPORT
##############################################################################
print('=' * 60)
print('FINAL SCALING REPORT')
print('=' * 60)
print()
print(f'{"Qubits":<8} {"Reg Dim":<10} {"Quantum":<16} {"Classical":<16} {"Advantage":<12} {"Seeds":<6}')
print('-' * 68)
for nq in sorted(RESULTS.keys()):
    r = RESULTS[nq]
    if r.get('failed'):
        print(f'{nq:<8} {2**(nq//3):<10} {"FAILED":<16} {"-":<16} {"-":<12} 0')
        continue
    dim = 2 ** (nq // 3)
    n_seeds = len(r['q_accs'])
    q_str = f"{r['mean_q']*100:.1f}% ±{np.std(r['q_accs'])*100:.1f}%" if n_seeds > 1 else f"{r['mean_q']*100:.1f}%"
    c_str = f"{r['mean_c']*100:.1f}% ±{np.std(r['c_accs'])*100:.1f}%" if n_seeds > 1 else f"{r['mean_c']*100:.1f}%"
    print(f'{nq:<8} {dim:<10} {q_str:<16} {c_str:<16} {r["advantage"]*100:+.1f}%       {n_seeds}')

# Statistical test for 18q (5 seeds)
print()
print('=' * 60)
print('STATISTICAL SIGNIFICANCE (18q, 5 seeds)')
print('=' * 60)
if len(RESULTS.get(18, {}).get('q_accs', [])) >= 3:
    from scipy import stats
    q_vals = RESULTS[18]['q_accs']
    c_vals = RESULTS[18]['c_accs']
    t_stat, p_value = stats.ttest_rel(q_vals, c_vals)
    diff = np.array(q_vals) - np.array(c_vals)
    cohens_d = np.mean(diff) / np.std(diff, ddof=1) if np.std(diff, ddof=1) > 0 else 0
    print(f'  Paired t-test: t={t_stat:.3f}, p={p_value:.4f}')
    print(f"  Cohen's d: {cohens_d:.2f}")
    print(f'  Mean advantage: {np.mean(diff)*100:+.1f}%')
    print(f'  Significant (p<0.05): {"YES ★" if p_value < 0.05 else "NO"}')
else:
    print('  Not enough seeds for statistical test')

# Scaling trend
print()
print('=' * 60)
print('SCALING VERDICT')
print('=' * 60)
valid_results = {k: v for k, v in RESULTS.items() if not v.get('failed')}
if len(valid_results) >= 3:
    sorted_nq = sorted(valid_results.keys())
    advantages = [valid_results[nq]['advantage'] for nq in sorted_nq]
    print(f'  Scaling curve: {", ".join(f"{nq}q:{a*100:+.1f}%" for nq, a in zip(sorted_nq, advantages))}')
    print()

    # Check monotonic increase
    increasing = all(advantages[i+1] > advantages[i] for i in range(len(advantages)-1))
    last_positive = advantages[-1] > 0.03
    any_significant = any(a > 0.05 for a in advantages)

    if increasing and last_positive:
        print('  ✓ ADVANTAGE GROWS MONOTONICALLY WITH QUBITS')
        print('  → Scaling thesis STRONGLY validated')
        print('  → Proceed to Month 2: real quantum hardware (IBM Quantum)')
    elif last_positive:
        print('  ~ ADVANTAGE EXISTS at highest qubit count')
        print('  → Scaling thesis partially validated')
        print('  → Quantum advantage emerges at sufficient scale')
    elif any_significant:
        print('  ? MIXED: Advantage exists but not consistently growing')
        print('  → May need different learning rates or more epochs at higher qubits')
    else:
        print('  ✗ NO CONSISTENT ADVANTAGE at any scale')
        print('  → Scaling thesis not validated')
        print('  → Consider fundamentally different circuit designs')
else:
    print('  Not enough qubit scales completed for trend analysis')

print()
print('Copyright (c) 2026 Saikrishna Garikipati. All Rights Reserved.')